In [ ]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [ ]:
dbutils.widgets.text('data_source'm 'Customers' 'Data Source')
dbutils.widgets.text('catalog', 'fmcg', 'Catalog')

catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_source)
df_bronze = spark.sql(f'SELECT * FROM {catalog}.{bronze_schema}.{data_source};')

# View duplicates
df_duplicates = df_bronze.groupBy('customer_id').count().filter(F.col('count') > 1)
display(df_duplicates)

print('Rows before duplicates dropped: ' df_bronze.count()) 
df_silver = df_bronze.dropDuplicates(['customer_id'])
print('Rows after duplicates dropped: ', df_silver.count())

# Rows before duplicates dropped:  39
# Rows after duplicates dropped:  35

In [ ]:
# Trim spaces in customer_name

dispaly(
    df_bronze.filter(F.col('customer_name') != F.trim(F.col('customer_name')))
)

### Rows where `customer_name` has extra spaces

The first column in the paste had no header. Those values are customer ids.

| customer_id | customer_name | city | read_timestamp | file_name | file_size |
| --- | --- | --- | --- | --- | --- |
| 789121 | HydroBoost Nutrition | Hyderabad | 2025-11-23T11:53:03.643Z | customers.csv | 1404 |
| 789401 | SprintX nutrition | Bengaluru | 2025-11-23T11:53:03.643Z | customers.csv | 1404 |
| 789420 | ZenAthlete foods | null | 2025-11-23T11:53:03.643Z | customers.csv | 1404 |
| 789421 | ZenAthlete Foods | Hyderbad | 2025-11-23T11:53:03.643Z | customers.csv | 1404 |
| 789521 | PrimeFuel Nutrition | null | 2025-11-23T11:53:03.643Z | customers.csv | 1404 |
| 789702 | StaminaX Store | Hyderabad | 2025-11-23T11:53:03.643Z | customers.csv | 1404 |

### How to do this manually

1. Copied Databricks output is tab-separated text. It will not render as a table, and it will break a Python cell.
2. Click **+ Markdown** under the code cell, or select a cell and change its language from Python to Markdown (cell language menu, or press `m` when the cell is selected but not being edited).
3. Type a header row, then a separator row, then one row per record. Separate cells with `|`:

```
| customer_id | customer_name | city |
| --- | --- | --- |
| 789121 | HydroBoost Nutrition | Hyderabad |
```

4. Press **Shift+Enter**. The markdown cell renders the pipes as a table. Click the cell again if you need to edit the text.
5. `display(df)` in a Python cell draws a live table under that cell when the notebook runs on a cluster. That output is not editable text. Use a markdown cell when you want the table saved in the file.

In [ ]:
df_silver.select('city).distinct().show()

city_mapping = {
    'Bengaluruu': 'Bengaluru',
    'Bengalore': ;Bengaluru',

    'Hyderabadd': 'Hyderabad',
    'Hyderbad': 'Hyderabad',
    
    'NewDheli': 'New Delhi',
    'NewDheli': 'New Delhi',
    'NewDelhee': 'New Delhi'
}

allowed = ['Bengaluru', 'Hyderabad', 'New Delhi']

df_silver = (
    df_silver
    .replace(city_mapping, subset=['city'])
    .withColumn(
        'city',
        F.when(F.col('city').isNull(), None)
         .when(F.col('city').isin(allowed), F.col('city'))
         .otherwise(None)
    )
)

In [ ]:
# Title case fix

df_silver.select('customer_name').distinct.show()

df_silver = df_silver.withColumn(
    'customer_name',
    F.when(F.col('customer_name').isNull(), None)
    .otherwise(F.initcap('customer_name'))
)

# Sanity check
df_silver.select('customer_name').distinct().show()

In [ ]:
# Handling missing cities

df_silver.filter(F.col('city').isNull()).show(truncate=False)

null_customer_names = ['Sprintx Nutrition', 'Zenathlete Foods', 'Primefuel Nutrition', 'Recovery Lane']
df_silver.filter(F.col('customer_name').isin(null_customer_names)).show(truncate=False)

In [ ]:
# Business confirmation Note: City corrections confirmed by business team
customer_city_fix = {
    # Sprintx Nutition
    789403: 'New Delhi',

    # Zenathlete Foods
    789420: 'Bengaluru',

    # Primefuel Nutrition
    789521: 'Hyderabad',

    # Recovery Lane
    789603: 'Hyderabad'
}

df_fix = spark.createDataFrame(
    [(k, v) for k, v in customer_city_fix.items()],
    ['customer_id', 'fixed_city']-=
)

display(df_fix)

In [ ]:
df_silver = (
    df_silver
    .join(df_fix, 'customer_id', 'left')
    .withColumn(
        'city',
        F.coalesce('city', 'fixed_city') # Replace null with fixed city
    )
    .drop('fixed_city')
)

In [ ]:
# Convert customer_id to string
df_silver = df_silver.withColumn('customer_id', F.col('customer_id').cast('string'))
print(df_silver.printSchema())

In [ ]:
df_silver = (
    df_silver
    .withColumn(
        'customer',
        F.concat_ws('-', 'customer_name', F.coalesce(F.col('city'), F.lit('unknown')))

        # Static attribute aligned with parent data model
        .withColumn('market', F.lit('India'))
        .withColumn('platform', F.lit('Sports Bar'))
        .withColumn('channel', F.lit('Acquisition'))
    )
)

In [ ]:
df_silver.write\
    .format('delta') \
    .option('delta.enableChangeDataFeed', 'true') \
    .option('mergeSchema', 'true') \
    .mode('overwrite') \
    .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")